In [18]:
# =========================
# DAG Validation (Pre-check)
# =========================
import networkx as nx
from typing import List, Tuple


def validate_dag_from_gexf(gexf_path: str, max_cycles: int = 3) -> None:
    """
    - GEXF 그래프가 DAG인지 검증
    - cycle 존재 시 예시 출력
    - 양방향 edge 여부 점검
    """
    print("=" * 80)
    print(f"[DAG CHECK] Loading graph: {gexf_path}")

    G = nx.read_gexf(gexf_path)

    # 노드/엣지 요약
    print(f"[INFO] nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    # DAG 여부
    is_dag = nx.is_directed_acyclic_graph(G)
    print(f"[CHECK] is_directed_acyclic_graph: {is_dag}")

    # 양방향(edge conflict) 검사
    bidirectional: List[Tuple[str, str]] = []
    for u, v in G.edges():
        if G.has_edge(v, u):
            bidirectional.append((u, v))

    if bidirectional:
        print(f"[WARN] bidirectional edges detected (count={len(bidirectional)//2}):")
        for u, v in bidirectional[:10]:
            print(f"  - {u} <-> {v}")
    else:
        print("[OK] no bidirectional edges")

    # cycle 검사
    if not is_dag:
        print("[ERROR] graph contains cycles")
        cycles = list(nx.simple_cycles(G))

        print(f"[INFO] number of cycles found: {len(cycles)}")

        for idx, cyc in enumerate(cycles[:max_cycles]):
            print(f"  cycle[{idx + 1}] length={len(cyc)}:")
            print("   -> " + " -> ".join(cyc + [cyc[0]]))

        if len(cycles) > max_cycles:
            print(f"  ... ({len(cycles) - max_cycles} more cycles omitted)")

    else:
        print("[OK] graph is a valid DAG")

    print("=" * 80)


In [14]:
import os
from typing import Dict, Optional, List, Tuple, Callable

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# (선택) GES_reg 생성에 필요
try:
    import pandas as pd
    from sklearn.preprocessing import StandardScaler
    _HAS_SKLEARN = True
except Exception:
    _HAS_SKLEARN = False


# =========================
# Settings
# =========================
BASE_DIR = "."
OUT_BASE = os.path.join(BASE_DIR, "layouts_all")

PATHS = {
    "NOTEARS": os.path.join(BASE_DIR, "graph_NOTEARS.gexf"),
    "GOLEM": os.path.join(BASE_DIR, "graph_GOLEM.gexf"),
    "PC": os.path.join(BASE_DIR, "graph_PC.gexf"),
    "GES_score": os.path.join(BASE_DIR, "graph_GES.gexf"),
    "GES_reg": os.path.join(BASE_DIR, "graph_GES_reg.gexf"),  # 없으면 아래에서 자동 생성 시도
}

# GES_reg 생성에 사용할 데이터 (없으면 GES_reg는 자동 생성/시각화 스킵)
DATA_PATH = os.path.join(BASE_DIR, "training_data_normalized.csv")
TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# 그림 출력
FIGSIZE = (16, 12)
DPI = 300
NODE_MIN = 260
NODE_MAX = 1400
ARROWSIZE = 14
FONT_NODE = 9
FONT_EDGE = 8
EDGE_W_MIN = 0.6
EDGE_W_MAX = 4.0

# edge label: 상위 |value| K개만 표시
TOPK_EDGE_LABELS = 30

# w=0 edge 제거 기준
EPS = 0.0

# 너무 빽빽하면 여기서 줄이기 (None이면 전부)
TOP_EDGES = None  # 예: 200

# 레이아웃 종류: "최대한 다양하게"
# - graphviz_*는 설치되어 있으면 자동 사용, 없으면 spring으로 fallback
LAYOUTS = [
    # NetworkX 기본/대표
    "spring",
    "kamada_kawai",
    "spectral",
    "circular",
    "shell",
    "random",
    "spiral",

    # DAG/계층형(논문에서 가장 읽기 좋음)
    "multipartite_topo",
    "bfs_layers",
    "radial_layers",

    # ForceAtlas2 스타일(라이브러리 없이 근사: spring 파라미터 변경)
    "spring_tight",
    "spring_loose",

    # Graphviz 계열(있으면 강력)
    "graphviz_dot",
    "graphviz_neato",
    "graphviz_fdp",
    "graphviz_sfdp",
    "graphviz_circo",
    "graphviz_twopi",

    # 대체 레이아웃: planar는 그래프가 planar가 아닐 수 있어 실패 가능 -> 안전 fallback 포함
    "planar_or_fallback",
]


# =========================
# Utils
# =========================
def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def _as_float(x) -> float:
    try:
        return float(x)
    except Exception:
        return 0.0


def load_gexf_as_digraph(path: str) -> nx.DiGraph:
    """MultiDiGraph 방지 + 속성 보존 + 노드명 str 통일"""
    G0 = nx.read_gexf(path)
    G = nx.DiGraph()

    for n, data in G0.nodes(data=True):
        G.add_node(str(n), **data)

    if isinstance(G0, nx.MultiDiGraph):
        for u, v, k, data in G0.edges(keys=True, data=True):
            u2, v2 = str(u), str(v)
            nd = dict(data)
            for attr in ["weight", "score", "w"]:
                if attr in nd:
                    nd[attr] = _as_float(nd.get(attr, 0.0))

            if not G.has_edge(u2, v2):
                G.add_edge(u2, v2, **nd)
            else:
                # 같은 (u,v)면 |attr| 큰 값 유지
                for attr in ["score", "weight", "w"]:
                    if attr in nd:
                        newv = _as_float(nd.get(attr, 0.0))
                        oldv = _as_float(G[u2][v2].get(attr, 0.0))
                        if abs(newv) > abs(oldv):
                            G[u2][v2][attr] = newv
    else:
        for u, v, data in G0.edges(data=True):
            u2, v2 = str(u), str(v)
            nd = dict(data)
            for attr in ["weight", "score", "w"]:
                if attr in nd:
                    nd[attr] = _as_float(nd.get(attr, 0.0))
            G.add_edge(u2, v2, **nd)

    G.remove_edges_from(list(nx.selfloop_edges(G)))
    return G


def pick_edge_attr_for_alg(alg: str, G: nx.DiGraph) -> Optional[str]:
    """PC는 구조만, GES_score는 score 우선, 나머지는 weight 우선"""
    if alg == "PC":
        return None

    if alg == "GES_score":
        candidates = ["score", "weight", "w"]
    else:
        candidates = ["weight", "score", "w"]

    edges = list(G.edges())
    if not edges:
        return None

    for attr in candidates:
        for u, v in edges[: min(200, len(edges))]:
            if attr in G[u][v]:
                return attr
    return None


def filter_edges(G: nx.DiGraph, edge_attr: Optional[str]) -> nx.DiGraph:
    """|w|<=EPS 제거 + (선택) TOP_EDGES로 상위 |w|만 유지"""
    H = nx.DiGraph()
    H.add_nodes_from(G.nodes(data=True))

    if edge_attr is None:
        # PC: 구조만. (원하면 여기에도 TOP_EDGES 같은 제한을 추가할 수 있음)
        for u, v, data in G.edges(data=True):
            if u != v:
                H.add_edge(u, v, **data)
        return H

    tmp = []
    for u, v, data in G.edges(data=True):
        if u == v:
            continue
        val = _as_float(data.get(edge_attr, 0.0))
        if abs(val) <= EPS:
            continue
        tmp.append((u, v, val, data))

    if TOP_EDGES is not None and len(tmp) > TOP_EDGES:
        tmp = sorted(tmp, key=lambda x: abs(x[2]), reverse=True)[:TOP_EDGES]

    for u, v, val, data in tmp:
        H.add_edge(u, v, **data)

    return H


def compute_node_sizes(G: nx.DiGraph) -> List[float]:
    """차수 기반 node size"""
    deg = {n: (G.in_degree(n) + G.out_degree(n)) for n in G.nodes()}
    if not deg:
        return []
    vals = np.array(list(deg.values()), dtype=float)
    vmin, vmax = float(vals.min()), float(vals.max())
    if vmax - vmin < 1e-12:
        return [(NODE_MIN + NODE_MAX) / 2 for _ in G.nodes()]
    out = []
    for n in G.nodes():
        t = (deg[n] - vmin) / (vmax - vmin)
        out.append(NODE_MIN + t * (NODE_MAX - NODE_MIN))
    return out


def compute_edge_widths(values: List[float]) -> List[float]:
    """|w| -> 굵기 (그래프 내부 스케일링)"""
    if not values:
        return []
    absvals = np.array([abs(v) for v in values], dtype=float)
    vmin, vmax = float(absvals.min()), float(absvals.max())
    if vmax - vmin < 1e-12:
        return [1.5 for _ in values]
    out = []
    for a in absvals:
        # sqrt로 극단값 완화
        t = (a - vmin) / (vmax - vmin)
        t = float(np.clip(t, 0.0, 1.0))
        t = np.sqrt(t)
        out.append(EDGE_W_MIN + t * (EDGE_W_MAX - EDGE_W_MIN))
    return out


def connected_components_positions(UG: nx.Graph, layout_fn: Callable[[nx.Graph], Dict]) -> Dict[str, Tuple[float, float]]:
    """컴포넌트별로 layout 후 x-offset으로 겹침 방지"""
    comps = list(nx.connected_components(UG))
    if not comps:
        return {}

    all_pos = {}
    x_offset = 0.0
    for comp in sorted(comps, key=len, reverse=True):
        sub = UG.subgraph(comp).copy()
        pos_c = layout_fn(sub)

        xs = [pos_c[n][0] for n in sub.nodes()] if sub.number_of_nodes() else [0.0]
        minx, maxx = float(min(xs)), float(max(xs))
        width = (maxx - minx) if (maxx - minx) > 1e-9 else 1.0

        for n, (x, y) in pos_c.items():
            all_pos[str(n)] = (float(x + x_offset), float(y))
        x_offset += width * 2.2

    return all_pos


def topo_levels(G: nx.DiGraph) -> Dict[str, int]:
    """DAG depth 레벨. DAG 아니면 in-degree fallback."""
    try:
        order = list(nx.topological_sort(G))
    except Exception:
        return {n: int(G.in_degree(n)) for n in G.nodes()}

    level = {n: 0 for n in G.nodes()}
    for v in order:
        preds = list(G.predecessors(v))
        if preds:
            level[v] = 1 + max(level[p] for p in preds)
    return level


def bfs_layers(G: nx.DiGraph) -> Dict[str, int]:
    """진입차수 0 노드들을 루트로 BFS 레벨"""
    roots = [n for n in G.nodes() if G.in_degree(n) == 0]
    if not roots:
        # fallback
        return topo_levels(G)

    UG = G.to_undirected()
    dist = {r: 0 for r in roots}
    from collections import deque
    q = deque(roots)
    while q:
        u = q.popleft()
        for v in UG.neighbors(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    # dist가 없는 노드는 0
    return {n: int(dist.get(n, 0)) for n in G.nodes()}


def safe_layout(G: nx.DiGraph, layout_name: str) -> Dict[str, Tuple[float, float]]:
    """
    레이아웃 계산에서 weight/score를 절대 사용하지 않음(negative weight 에러 차단).
    graphviz_*는 가능하면 사용, 안 되면 spring fallback.
    """
    UG = G.to_undirected()

    def spring_default(H): return nx.spring_layout(H, seed=42)
    def spring_tight(H): return nx.spring_layout(H, seed=42, k=0.08, iterations=300)
    def spring_loose(H): return nx.spring_layout(H, seed=42, k=0.35, iterations=200)
    def kk(H): return nx.kamada_kawai_layout(H)
    def spectral(H): return nx.spectral_layout(H)
    def circular(H): return nx.circular_layout(H)
    def shell(H): return nx.shell_layout(H)
    def random(H): return nx.random_layout(H, seed=42)
    def spiral(H): return nx.spiral_layout(H)

    def multipartite_topo(_):
        levels = topo_levels(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        return nx.multipartite_layout(G, subset_key="subset")

    def bfs_layers_layout(_):
        levels = bfs_layers(G)
        for n, lv in levels.items():
            G.nodes[n]["subset"] = int(lv)
        return nx.multipartite_layout(G, subset_key="subset")

    def radial_layers_layout(_):
        # shell_layout로 레벨별 원형 배치
        levels = topo_levels(G)
        max_lv = max(levels.values()) if levels else 0
        shells = []
        for lv in range(max_lv + 1):
            shells.append([n for n in G.nodes() if levels.get(n, 0) == lv])
        # 빈 shell 제거(필수)
        shells = [s for s in shells if s]
        return nx.shell_layout(UG, nlist=shells)

    def graphviz_layout_prog(prog: str):
        from networkx.drawing.nx_pydot import graphviz_layout
        pos = graphviz_layout(G, prog=prog)
        return {str(k): (float(v[0]), float(v[1])) for k, v in pos.items()}

    def planar_or_fallback(H):
        try:
            pos = nx.planar_layout(H)
            return {str(k): (float(v[0]), float(v[1])) for k, v in pos.items()}
        except Exception:
            return spring_default(H)

    # 컴포넌트별 배치 + offset
    try:
        if layout_name == "spring":
            return connected_components_positions(UG, spring_default)
        if layout_name == "spring_tight":
            return connected_components_positions(UG, spring_tight)
        if layout_name == "spring_loose":
            return connected_components_positions(UG, spring_loose)
        if layout_name == "kamada_kawai":
            return connected_components_positions(UG, kk)
        if layout_name == "spectral":
            return connected_components_positions(UG, spectral)
        if layout_name == "circular":
            return connected_components_positions(UG, circular)
        if layout_name == "shell":
            return connected_components_positions(UG, shell)
        if layout_name == "random":
            return connected_components_positions(UG, random)
        if layout_name == "spiral":
            return connected_components_positions(UG, spiral)
        if layout_name == "multipartite_topo":
            return multipartite_topo(UG)
        if layout_name == "bfs_layers":
            return bfs_layers_layout(UG)
        if layout_name == "radial_layers":
            return radial_layers_layout(UG)
        if layout_name == "planar_or_fallback":
            return connected_components_positions(UG, planar_or_fallback)

        if layout_name.startswith("graphviz_"):
            prog = layout_name.replace("graphviz_", "")
            try:
                # graphviz는 컴포넌트 offset을 안 하되, 실패하면 fallback
                return graphviz_layout_prog(prog)
            except Exception:
                return connected_components_positions(UG, spring_default)

        return connected_components_positions(UG, spring_default)
    except Exception:
        return connected_components_positions(UG, spring_default)


# =========================
# (Optional) Build GES_reg if missing
# =========================
def load_numeric_X_for_regression(data_path: str):
    if not _HAS_SKLEARN:
        raise RuntimeError("pandas / scikit-learn이 없어 GES_reg를 생성할 수 없습니다.")

    df = pd.read_csv(data_path, low_memory=False)
    drop_cols = [c for c in df.columns if c in TARGET_CANDIDATES]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    col_idx = {c: i for i, c in enumerate(col_names)}

    X = df.values.astype(float)
    X = StandardScaler().fit_transform(X)  # 표준화 OLS
    return X, col_idx


def ensure_ges_reg_graph(ges_score_gexf: str, out_ges_reg_gexf: str, data_path: str) -> bool:
    if os.path.exists(out_ges_reg_gexf):
        return True

    if not os.path.exists(ges_score_gexf):
        print(f"[SKIP] GES_reg: missing GES_score graph: {ges_score_gexf}")
        return False

    if not os.path.exists(data_path):
        print(f"[SKIP] GES_reg: missing DATA_PATH: {data_path}")
        return False

    if not _HAS_SKLEARN:
        print("[SKIP] GES_reg: pandas/scikit-learn not available.")
        return False

    X, col_idx = load_numeric_X_for_regression(data_path)
    G = load_gexf_as_digraph(ges_score_gexf)  # 구조는 score 그래프 사용

    for child in list(G.nodes()):
        if child not in col_idx:
            continue
        parents = [p for p in G.predecessors(child) if p in col_idx]
        if not parents:
            continue

        y = X[:, col_idx[child]]
        Xp = X[:, [col_idx[p] for p in parents]]
        coef, *_ = np.linalg.lstsq(Xp, y, rcond=None)
        for p, w in zip(parents, coef):
            G[p][child]["weight"] = float(w)

    nx.write_gexf(G, out_ges_reg_gexf)
    print(f"[BUILD] created: {out_ges_reg_gexf}")
    return True


# =========================
# Drawing
# =========================
def draw_graph(
    alg: str,
    G: nx.DiGraph,
    edge_attr: Optional[str],
    layout_name: str,
    out_dir: str
) -> None:
    ensure_dir(out_dir)

    # edge 필터 적용(|w|<=EPS 제거 + TOP_EDGES)
    H = filter_edges(G, edge_attr=edge_attr)
    edges = list(H.edges())
    if len(edges) == 0:
        print(f"[SKIP] {alg} | {layout_name}: no edges after filtering (edge_attr={edge_attr})")
        return

    pos = safe_layout(H, layout_name)
    # pos에 누락된 노드가 있으면 spring으로 채움
    if len(pos) < H.number_of_nodes():
        missing = [n for n in H.nodes() if n not in pos]
        fb = nx.spring_layout(H.to_undirected(), seed=42)
        for n in missing:
            pos[n] = (float(fb[n][0]), float(fb[n][1]))

    # values
    if edge_attr is None:
        values = [0.0 for _ in edges]
        edge_colors = ["0.35" for _ in edges]  # PC는 회색
    else:
        values = [_as_float(H[u][v].get(edge_attr, 0.0)) for u, v in edges]
        edge_colors = ["tab:blue" if v > 0 else "tab:red" if v < 0 else "0.55" for v in values]

    widths = compute_edge_widths(values)

    # edge labels: top |w|
    edge_labels = {}
    if edge_attr is not None and TOPK_EDGE_LABELS > 0:
        order = np.argsort([abs(v) for v in values])[::-1]
        for idx in order[: min(TOPK_EDGE_LABELS, len(order))]:
            u, v = edges[idx]
            edge_labels[(u, v)] = f"{values[idx]:.2f}"

    node_sizes = compute_node_sizes(H)

    title_suffix = "structure only" if edge_attr is None else edge_attr
    title = f"{alg} | {layout_name} | {title_suffix} | EPS={EPS} | TOP_EDGES={TOP_EDGES}"

    out_png = os.path.join(out_dir, f"{alg}__{layout_name}.png")

    plt.figure(figsize=FIGSIZE, dpi=DPI)
    plt.title(title)

    nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color="white", edgecolors="black", linewidths=0.8)
    nx.draw_networkx_labels(H, pos, font_size=FONT_NODE)

    nx.draw_networkx_edges(
        H, pos,
        edge_color=edge_colors,
        width=widths if widths else 1.2,
        arrows=True,
        arrowsize=ARROWSIZE,
        alpha=0.85,
        connectionstyle="arc3,rad=0.05",
    )

    if edge_labels:
        nx.draw_networkx_edge_labels(H, pos, edge_labels=edge_labels, font_size=FONT_EDGE)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

    print(f"[DONE] {alg} | {layout_name} -> {out_png}")


def main():
    ensure_dir(OUT_BASE)

    # GES_reg가 없으면 생성 시도
    ensure_ges_reg_graph(PATHS["GES_score"], PATHS["GES_reg"], DATA_PATH)

    # 그래프 로드
    graphs: Dict[str, nx.DiGraph] = {}
    for alg, p in PATHS.items():
        if not os.path.exists(p):
            print(f"[SKIP] Missing: {p}")
            continue
        graphs[alg] = load_gexf_as_digraph(p)

    if not graphs:
        raise FileNotFoundError("No graphs to draw. Check BASE_DIR/PATHS.")

    # 레이아웃 다양하게 전부 저장
    for alg, G in graphs.items():
        edge_attr = pick_edge_attr_for_alg(alg, G)
        out_dir = os.path.join(OUT_BASE, alg)

        for layout_name in LAYOUTS:
            draw_graph(
                alg=alg,
                G=G,
                edge_attr=edge_attr,
                layout_name=layout_name,
                out_dir=out_dir
            )

    print(f"[ALL DONE] outputs -> {OUT_BASE}")
    print("Layouts used:", ", ".join(LAYOUTS))
    print(f"Zero-edge filter: abs(value) <= {EPS} removed (when edge_attr is not None)")


if __name__ == "__main__":
    main()


[DONE] NOTEARS | spring -> .\layouts_all\NOTEARS\NOTEARS__spring.png
[DONE] NOTEARS | kamada_kawai -> .\layouts_all\NOTEARS\NOTEARS__kamada_kawai.png
[DONE] NOTEARS | spectral -> .\layouts_all\NOTEARS\NOTEARS__spectral.png
[DONE] NOTEARS | circular -> .\layouts_all\NOTEARS\NOTEARS__circular.png
[DONE] NOTEARS | shell -> .\layouts_all\NOTEARS\NOTEARS__shell.png
[DONE] NOTEARS | random -> .\layouts_all\NOTEARS\NOTEARS__random.png
[DONE] NOTEARS | spiral -> .\layouts_all\NOTEARS\NOTEARS__spiral.png
[DONE] NOTEARS | multipartite_topo -> .\layouts_all\NOTEARS\NOTEARS__multipartite_topo.png
[DONE] NOTEARS | bfs_layers -> .\layouts_all\NOTEARS\NOTEARS__bfs_layers.png
[DONE] NOTEARS | radial_layers -> .\layouts_all\NOTEARS\NOTEARS__radial_layers.png
[DONE] NOTEARS | spring_tight -> .\layouts_all\NOTEARS\NOTEARS__spring_tight.png
[DONE] NOTEARS | spring_loose -> .\layouts_all\NOTEARS\NOTEARS__spring_loose.png


C:\Users\User\AppData\Local\Temp\ipykernel_27484\3360338153.py:328: DeprecationWarning: nx.nx_pydot.graphviz_layout depends on the pydot package, which has known issues and is not actively maintained. Consider using nx.nx_agraph.graphviz_layout instead.

See https://github.com/networkx/networkx/issues/5723
  pos = graphviz_layout(G, prog=prog)


[DONE] NOTEARS | graphviz_dot -> .\layouts_all\NOTEARS\NOTEARS__graphviz_dot.png
[DONE] NOTEARS | graphviz_neato -> .\layouts_all\NOTEARS\NOTEARS__graphviz_neato.png
[DONE] NOTEARS | graphviz_fdp -> .\layouts_all\NOTEARS\NOTEARS__graphviz_fdp.png
[DONE] NOTEARS | graphviz_sfdp -> .\layouts_all\NOTEARS\NOTEARS__graphviz_sfdp.png
[DONE] NOTEARS | graphviz_circo -> .\layouts_all\NOTEARS\NOTEARS__graphviz_circo.png
[DONE] NOTEARS | graphviz_twopi -> .\layouts_all\NOTEARS\NOTEARS__graphviz_twopi.png
[DONE] NOTEARS | planar_or_fallback -> .\layouts_all\NOTEARS\NOTEARS__planar_or_fallback.png
[DONE] GOLEM | spring -> .\layouts_all\GOLEM\GOLEM__spring.png
[DONE] GOLEM | kamada_kawai -> .\layouts_all\GOLEM\GOLEM__kamada_kawai.png
[DONE] GOLEM | spectral -> .\layouts_all\GOLEM\GOLEM__spectral.png
[DONE] GOLEM | circular -> .\layouts_all\GOLEM\GOLEM__circular.png
[DONE] GOLEM | shell -> .\layouts_all\GOLEM\GOLEM__shell.png
[DONE] GOLEM | random -> .\layouts_all\GOLEM\GOLEM__random.png
[DONE] GOLE